In [13]:
 %run ../setup/01_config

Box(children=(Label(value='Catalog'), Text(value='dbr_dev')))

Box(children=(Label(value='Schema'), Text(value='bronze')))

In [14]:
from pyspark.sql.functions import col, current_timestamp

In [15]:
dbutils.widgets.dropdown(
    "trigger_type", "availableNow", ["availableNow", "once", "processingTime"]
)
trigger_type = dbutils.widgets.get("trigger_type")
print(f"Using trigger type: {trigger_type}")


Box(children=(Label(value='trigger_type'), Dropdown(options=('availableNow', 'once', 'processingTime'), value=…

Using trigger type: availableNow


In [16]:
BOOTSTRAP_SERVERS = "pkc-56d1g.eastus.azure.confluent.cloud:9092"
TOPIC_NAME = "orders_event"

api_key = dbutils.secrets.get(scope="confluent-scope", key="api-key")
api_secret = dbutils.secrets.get(scope="confluent-scope", key="api-secret")


In [19]:
kafka_options = {
    "kafka.bootstrap.servers": BOOTSTRAP_SERVERS,
    "subscribe": TOPIC_NAME,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required "
        f'username="{api_key}" password="{api_secret}";'
    ),
    "startingOffsets": "earliest",
}

In [20]:
df_raw = spark.readStream.format("kafka").options(**kafka_options).load()

df_bronze = (
    df_raw
    .withColumn("raw_json", col("value").cast("string"))
    .withColumn("_ingest_timestamp", current_timestamp())
    .select("raw_json", "topic", "partition", "offset", "timestamp", "_ingest_timestamp")
)

trigger_options = {
    "availableNow": {"availableNow": True},
    "once": {"once": True},
    "processingTime": {"processingTime": "10 seconds"},
}

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.brz_orders_events"

query = (
    df_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", ORDERS_CHECKPOINT_LOCATION)
    .trigger(**trigger_options[trigger_type])
    .toTable(BRONZE_TABLE)
)
query.awaitTermination()

print(f"Ingestion complete into {BRONZE_TABLE}")

Ingestion complete into dbr_dev.bronze.brz_orders_events
